In [3]:
import os
import mne
import numpy as np
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

mne.set_log_level('WARNING')

In [4]:
def subject_hand_classifier(subjects, tmin=0, tmax=2, n_csp=2, test_size=0.3, random_state=42):
    event_id = {'T1': 1, 'T2': 2}
    results = {}
    
    for subj in subjects:
        print(f"\nProcessing {subj}...")
        subj_files = [f"{subj}/{f}" for f in os.listdir(subj) if f.endswith('.edf')]
        
        X_subj, y_subj = [], []
        for fpath in subj_files:
            raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
            events, _ = mne.events_from_annotations(raw)
            existing_ids = [eid for eid in [1, 2] if eid in events[:, -1]]
            if len(existing_ids) < 2:
                continue
            
            event_id_run = {k: v for k, v in event_id.items() if v in existing_ids}
            epochs = mne.Epochs(raw, events, event_id=event_id_run,
                                tmin=tmin, tmax=tmax, baseline=None,
                                preload=True, verbose=False)
            keep_mask = np.isin(epochs.events[:, -1], existing_ids)
            epochs = epochs[keep_mask]
            X_subj.append(epochs.get_data())
            y_subj.append(epochs.events[:, -1] - 1)
        
        if not X_subj:
            print(f"No valid runs for {subj}, skipping.")
            continue
        
        X_subj = np.concatenate(X_subj, axis=0)
        y_subj = np.concatenate(y_subj, axis=0)
        
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        X_csp = csp.fit_transform(X_subj, y_subj)
        X_scaled = StandardScaler().fit_transform(X_csp)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_subj, test_size=test_size,
            random_state=random_state, stratify=y_subj
        )
        
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        
        print(f"{subj} Test Accuracy: {acc:.3f}")
        print(f"{subj} Confusion Matrix:\n{cm}")
        results[subj] = {"accuracy": acc, "confusion_matrix": cm}
    
    return results

def subject_foot_classifier(subjects, tmin=0, tmax=2, n_csp=2, test_size=0.3, random_state=42):
    results = {}
    
    for subj in subjects:
        print(f"\nProcessing {subj}...")
        subj_files = [f"{subj}/{f}" for f in os.listdir(subj) if f.endswith('.edf')]
        
        X_subj, y_subj = [], []
        for fpath in subj_files:
            raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
            descs = raw.annotations.description
            # Only use runs that contain both T1 (LF) and T2 (RF)
            available = {d: i+1 for i, d in enumerate(['T1','T2']) if d in descs}
            if len(available) < 2:
                continue
            
            events, _ = mne.events_from_annotations(raw, event_id=available)
            # Map numeric events to 0 (LF) and 1 (RF)
            y_vals = np.array([0 if ev==available['T1'] else 1 for ev in events[:, -1]])
            
            epochs = mne.Epochs(raw, events, event_id=available,
                                tmin=tmin, tmax=tmax, baseline=None,
                                preload=True, verbose=False)
            X_subj.append(epochs.get_data())
            y_subj.append(y_vals)
        
        if not X_subj:
            print(f"No valid runs for {subj}, skipping.")
            continue
        
        X_subj = np.concatenate(X_subj, axis=0)
        y_subj = np.concatenate(y_subj, axis=0)
        
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        X_csp = csp.fit_transform(X_subj, y_subj)
        X_scaled = StandardScaler().fit_transform(X_csp)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_subj, test_size=test_size,
            random_state=random_state, stratify=y_subj
        )
        
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        
        print(f"{subj} Test Accuracy: {acc:.3f}")
        print(f"{subj} Confusion Matrix:\n{cm}")
        results[subj] = {"accuracy": acc, "confusion_matrix": cm}
    
    return results

In [5]:
hand_test_results = subject_hand_classifier(['S004'])
foot_test_results = subject_foot_classifier(['S004'])


Processing S004...
S004 Test Accuracy: 0.951
S004 Confusion Matrix:
[[52  2]
 [ 2 25]]

Processing S004...
S004 Test Accuracy: 0.833
S004 Confusion Matrix:
[[24  3]
 [ 6 21]]
